In [3]:
import mne
import numpy as np
import pandas as pd
import gc
from pathlib import Path

# ══════════════════════════════════════════════════════════════════════════════
# Configuratie
# ══════════════════════════════════════════════════════════════════════════════
base_dir   = Path(r"\\vs03.herseninstituut.knaw.nl\VS03-SandC-2\raw\bnbd\Data\eeg\NSR")
output_dir = Path(r"C:\Users\zafar\Documents\bnbd_output2")
output_dir.mkdir(exist_ok=True)

MAX_PARTICIPANTS = 2

EEG_CH = ['EEG L psg-lp', 'EEG R psg-lp']
EMG_CH = ['EEG L psg-emg', 'EEG R psg-emg']
MOV_CH = ['dX', 'dY', 'dZ']
ALL_CH = EEG_CH + EMG_CH + MOV_CH

SFREQ         = 256.0
WIN_SEC       = 1.0
STEP_SEC      = 0.5
WIN_SAMP      = int(WIN_SEC  * SFREQ)
STEP_SAMP     = int(STEP_SEC * SFREQ)
CHUNK_MINUTES = 5
CHUNK_SAMP    = int(CHUNK_MINUTES * 60 * SFREQ)

FREQS = np.arange(0.5, 35.5, 0.5)

BANDS = {
    'delta': (0.5,  4.0),
    'theta': (4.0,  8.0),
    'alpha': (8.0,  13.0),
    'beta':  (13.0, 35.0),
}

ROLLING_SEC            = 60.0
AROUSAL_FREQ_THRESHOLD = 8.0
AROUSAL_MIN_DUR        = 3.0
AROUSAL_MAX_DUR        = 30.0


# ══════════════════════════════════════════════════════════════════════════════
# Morlet wavelet (van supervisor)
# ══════════════════════════════════════════════════════════════════════════════
def compute_morlet_tf(signal, srate, freqs, n_cycles=None, L2normalize=False):
    freqs = np.asarray(freqs)
    if n_cycles is None:
        n_cycles_arr = np.maximum(3.0, freqs / 2.0)
    elif np.isscalar(n_cycles):
        n_cycles_arr = np.full(len(freqs), float(n_cycles))
    else:
        n_cycles_arr = np.asarray(n_cycles, dtype=float)

    n_samples  = len(signal)
    signal     = signal - np.mean(signal)
    signal_fft = np.fft.fft(signal)
    fft_freqs  = np.fft.fftfreq(n_samples, d=1.0 / srate)

    power = np.empty((len(freqs), n_samples), dtype=np.float32)
    for i, freq in enumerate(freqs):
        sigma_f     = freq / n_cycles_arr[i]
        wavelet_fft = np.exp(-0.5 * ((fft_freqs - freq) / sigma_f) ** 2)
        if L2normalize:
            wavelet_fft /= np.sqrt(np.sum(wavelet_fft ** 2))
        analytic = np.fft.ifft(signal_fft * wavelet_fft)
        power[i] = np.abs(analytic) ** 2

    return power  # shape: (n_freqs, n_samples)


def band_mean(power, freqs, fmin, fmax):
    mask = (freqs >= fmin) & (freqs <= fmax)
    return power[mask, :].mean(axis=0)


# ══════════════════════════════════════════════════════════════════════════════
# Fase 1 — Load & preprocess
# ══════════════════════════════════════════════════════════════════════════════
def load_night(edf_file):
    raw = mne.io.read_raw_edf(edf_file, preload=False, verbose=False)
    raw.pick(ALL_CH)
    raw.load_data(verbose=False)
    raw._data = raw._data.astype(np.float64)

    # Volt naar microvolt
    for ch in EEG_CH + EMG_CH:
        idx = raw.ch_names.index(ch)
        raw._data[idx] *= 1e6

    raw.filter(l_freq=0.5, h_freq=35.0, picks=EEG_CH, verbose=False)
    h_emg = min(100.0, SFREQ / 2 - 1)
    raw.filter(l_freq=10.0, h_freq=h_emg, picks=EMG_CH, verbose=False)
    raw.apply_function(lambda x: x - np.mean(x), picks=MOV_CH, verbose=False)

    return raw


def preprocess_signals(raw):
    signals = {}
    for ch in ALL_CH:
        signals[ch] = raw.get_data(picks=ch)[0]
    return signals


# ══════════════════════════════════════════════════════════════════════════════
# Fase 2 — Feature extractie via Morlet (chunked)
# ══════════════════════════════════════════════════════════════════════════════
def extract_features_morlet(signals, n_total):
    starts = np.arange(0, n_total - WIN_SAMP + 1, STEP_SAMP)
    df     = pd.DataFrame({'time_sec': starts / SFREQ})

    # ── EEG ──────────────────────────────────────────────────────────────────
    for ch in EEG_CH:
        tag    = 'L' if 'L' in ch else 'R'
        signal = signals[ch]
        print(f"    Morlet: {ch}")

        band_ts = {b: np.zeros(n_total, dtype=np.float64) for b in BANDS}

        offset = 0
        while offset < n_total:
            end   = min(offset + CHUNK_SAMP, n_total)
            chunk = signal[offset:end].astype(np.float64)
            power = compute_morlet_tf(chunk, srate=SFREQ, freqs=FREQS,
                                      L2normalize=True)
            for band_name, (fmin, fmax) in BANDS.items():
                band_ts[band_name][offset:end] = band_mean(power, FREQS, fmin, fmax)
            del power, chunk
            offset += CHUNK_SAMP

        for band_name in BANDS:
            col_vals = []
            for s in range(0, n_total - WIN_SAMP + 1, STEP_SAMP):
                col_vals.append(float(band_ts[band_name][s:s + WIN_SAMP].mean()))
            df[f'eeg_{tag}_{band_name}'] = col_vals

        rms_vals, ll_vals = [], []
        for s in range(0, n_total - WIN_SAMP + 1, STEP_SAMP):
            seg = signal[s:s + WIN_SAMP]
            rms_vals.append(float(np.sqrt(np.mean(seg ** 2))))
            ll_vals.append(float(np.sum(np.abs(np.diff(seg)))))

        df[f'eeg_{tag}_rms']             = rms_vals
        df[f'eeg_{tag}_line_length']     = ll_vals
        fast = df[f'eeg_{tag}_alpha'] + df[f'eeg_{tag}_beta']
        slow = df[f'eeg_{tag}_delta'] + df[f'eeg_{tag}_theta'] + 1e-12
        df[f'eeg_{tag}_fast_slow_ratio'] = fast / slow

        del band_ts

    # ── EMG ──────────────────────────────────────────────────────────────────
    for ch in EMG_CH:
        tag    = 'L' if 'L' in ch else 'R'
        signal = signals[ch]
        rms_vals = []
        for s in range(0, n_total - WIN_SAMP + 1, STEP_SAMP):
            seg = signal[s:s + WIN_SAMP]
            rms_vals.append(float(np.sqrt(np.mean(seg ** 2))))
        df[f'emg_{tag}_rms'] = rms_vals

    # ── Beweging ──────────────────────────────────────────────────────────────
    for ch in MOV_CH:
        axis   = ch[-1].lower()
        signal = signals[ch]
        rms_vals = []
        for s in range(0, n_total - WIN_SAMP + 1, STEP_SAMP):
            seg = signal[s:s + WIN_SAMP]
            rms_vals.append(float(np.sqrt(np.mean(seg ** 2))))
        df[f'mov_{axis}_rms'] = rms_vals

    return df


# ══════════════════════════════════════════════════════════════════════════════
# Fase 3 — Local baseline normalisatie
# ══════════════════════════════════════════════════════════════════════════════
def normalise_features(df):
    roll_rows = int(ROLLING_SEC / STEP_SEC)  # 120 rijen

    eeg_feature_cols = [
        c for c in df.columns
        if c.startswith('eeg_') and not c.endswith('_z')
    ]

    df_norm = df.copy()

    for col in eeg_feature_cols:
        roll_med = df[col].rolling(roll_rows, min_periods=1, center=False).median()
        roll_mad = (
            df[col]
            .rolling(roll_rows, min_periods=1, center=False)
            .apply(lambda x: np.median(np.abs(x - np.median(x))), raw=True)
        )
        z = (df[col] - roll_med) / (roll_mad.clip(lower=1e-3) + 1e-6)
        df_norm[f'{col}_z'] = z.clip(-10, 10)

    z_cols = [f'{c}_z' for c in eeg_feature_cols]
    df_norm['activation_score'] = df_norm[z_cols].mean(axis=1)

    return df_norm


# ══════════════════════════════════════════════════════════════════════════════
# Hoofdloop
# ══════════════════════════════════════════════════════════════════════════════
for participant_folder in sorted(base_dir.glob("bnbd_nsr_?????"))[:MAX_PARTICIPANTS]:
    pid = participant_folder.name.split("_")[-1]

    edf_file = (
        participant_folder
        / f"bnbd_nsr_{pid}_T0_N1"
        / "sleepArchitecture"
        / f"bnbd_nsr_{pid}_T0_N1_psg.edf"
    )

    if not edf_file.exists():
        print(f"[{pid}] Niet gevonden, overgeslagen.")
        continue

    out_path = output_dir / f"features_{pid}.csv"
    if out_path.exists():
        print(f"[{pid}] Al verwerkt, overgeslagen.")
        continue

    try:
        print(f"\n[{pid}] ── Fase 1: laden & preprocessen...")
        raw     = load_night(edf_file)
        signals = preprocess_signals(raw)
        n_total = raw.n_times
        del raw
        gc.collect()

        print(f"[{pid}] ── Fase 2: feature extractie (Morlet)...")
        df_feat = extract_features_morlet(signals, n_total)
        del signals
        gc.collect()

        print(f"[{pid}] ── Fase 3: normalisatie...")
        df_norm = normalise_features(df_feat)
        del df_feat
        gc.collect()

        df_norm.to_csv(out_path, index=False)
        print(f"[{pid}] Opgeslagen: {out_path.name}  ({len(df_norm)} vensters)")
        del df_norm
        gc.collect()

    except MemoryError:
        print(f"[{pid}] MemoryError — overgeslagen.")
        gc.collect()
        continue

print("\nFase 1–3 klaar.")

[00881] Niet gevonden, overgeslagen.

[01272] ── Fase 1: laden & preprocessen...
[01272] ── Fase 2: feature extractie (Morlet)...
    Morlet: EEG L psg-lp
    Morlet: EEG R psg-lp
[01272] ── Fase 3: normalisatie...
[01272] Opgeslagen: features_01272.csv  (53079 vensters)

Fase 1–3 klaar.
